# CVE Prioritization with Confidence-Weighted Learning-to-Rank

## Overview

This notebook implements confidence-weighted weak supervision for CVE prioritization:
- **Weak Labels**: Graded labels (0-3) constructed from KEV, EPSS, ATT&CK, Healthcare signals
- **Confidence Weights**: Each label has a confidence score reflecting its trustworthiness
- **LambdaRank Training**: LightGBM uses confidence weights to prioritize reliable labels
- **Comparison Study**: Evaluates against baselines and alternative models (DiffusionRank, RGCN, Ensemble)

### Architecture
- **Modules**: All functions migrated to `src/` for reusability
- **Notebook**: Orchestration and visualization only
- **GPU Support**: Automatic device detection (MPS/CUDA/CPU)

### Sections
1. Setup & Data Loading
2. EDA
3. Feature Engineering
4. Weak Label Construction
5. Temporal Split (Train/Val/Test)
6. Model Training (LTR + Comparison Models)
7. Evaluation & Comparison
8. Explainability (Feature Importance, SHAP)
9. Results & Conclusions

## 1. Setup & Configuration

In [ ]:
"""
CVE Prioritization
==================
Uses modular functions from src/ packages
Configuration loaded from YAML files (version-controlled)
"""

import os
import sys
import warnings
import pickle
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
# matplotlib replaced with plotly (see src/visualization modules)
import lightgbm as lgb

# Setup paths
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))
os.chdir(project_root)

# ============================================================
# CENTRALIZED CONFIGURATION (YAML-based, version-controlled)
# ============================================================
# Config files: config/experiments/{default,debug,production}.yaml
#
# To switch profiles:
#   exp_cfg.set_profile('debug')  # Fast iteration
#   exp_cfg.set_profile('production')  # Full data
# ============================================================
import config.experiment_config as exp_cfg
cfg = exp_cfg.cfg

# Import project modules
from src.core.cve_database import CVEDatabase
from src.features.labeling import build_weak_labels, print_label_diagnostics
from src.utils.temporal import make_temporal_splits
from src.models.ltr import prepare_ranking_data, train_lambdarank
from src.models.baselines import compute_cvss_only_scores, compute_heuristic_scores, compute_legacy_label_scores
from src.evaluation.metrics import evaluate_ranking, ndcg_at_k, precision_at_k
from src.visualization.explainability import (
    plot_feature_importance_comparison,
    plot_shap_summary,
    explain_individual_predictions
)
from src.utils.device_manager import get_device_manager

# GPU device manager for comparison models
device_manager = get_device_manager()

# Suppress warnings
warnings.filterwarnings('ignore')

# === CONFIGURATION FROM YAML ===
FEATURE_COLS = cfg.feature_cols
SPLIT_DATE = cfg.data.train_test_split_date
MODEL_PATH = cfg.model_path
RANDOM_SEED = 42
VAL_WEEKS = 12

# Print config summary
cfg.print_summary()

print("\n" + "=" * 70)
print("CVE PRIORITIZATION - FULL-SCALE TRAINING")
print("=" * 70)
print(f"Profile: {cfg._profile}")
print(f"Split date: {SPLIT_DATE}")
print(f"Validation weeks: {VAL_WEEKS}")
print(f"Features: {len(FEATURE_COLS)}")
print(f"Model output: {MODEL_PATH}")
print(f"Training device: {cfg.training_device}")
print("=" * 70)

## 2. Data Loading

In [ ]:
# Load data from CVEDatabase
db = CVEDatabase()
print(f"Connected to database: {db.db_path}")

# Load and merge CVEs with enrichments
cves_df = pd.read_sql("SELECT * FROM cves", db.conn)
enrichments_df = pd.read_sql("SELECT * FROM enrichments", db.conn)
df = cves_df.merge(enrichments_df, on='cve_id', how='left')

print(f"Total CVEs: {len(df):,}")
print(f"Date range: {pd.to_datetime(df['published']).min().date()} to {pd.to_datetime(df['published']).max().date()}")

# Convert dates
df['published'] = pd.to_datetime(df['published'])
df['modified'] = pd.to_datetime(df['modified'], errors='coerce')

print(f"\nEnrichment coverage:")
print(f"  KEV: {df['kev_flag'].notna().sum():,} ({100*df['kev_flag'].notna().sum()/len(df):.1f}%)")
print(f"  EPSS: {df['epss_score'].notna().sum():,} ({100*df['epss_score'].notna().sum()/len(df):.1f}%)")
print(f"  ATT&CK: {df['attack_technique_count'].notna().sum():,} ({100*df['attack_technique_count'].notna().sum()/len(df):.1f}%)")
print(f"  Healthcare: {df['is_healthcare'].notna().sum():,} ({100*df['is_healthcare'].notna().sum()/len(df):.1f}%)")

## 3 Exploratory Data Analysis (EDA)

Interactive visualizations to understand CVE data patterns:
- 📊 Temporal trends (CVEs over time)
- 📈 CVSS score distribution
- 🎯 KEV vs Non-KEV comparison
- 🔗 ATT&CK technique coverage
- 🏷️ Priority label distribution

In [ ]:
# Import EDA visualization functions
from src.visualization.eda import plot_all_eda

# Run all EDA visualizations
plot_all_eda(df, recent_months=24)

## 4. Feature Engineering

In [ ]:
# Feature Engineering using modular function
# Force reload to pick up latest module changes
import importlib
import src.features.engineering
importlib.reload(src.features.engineering)

from src.features.engineering import build_features, normalize_features, get_default_feature_cols

# =============================================================================
# Feature Engineering - Controlled by YAML Configuration
# =============================================================================
# Settings in config/experiments/{profile}.yaml:
#   - default.yaml: audit=true, plot=true (full diagnostics)
#   - debug.yaml: audit=true, plot=false (fast iteration)
#   - production.yaml: audit=false, plot=false (silent deployment)
# =============================================================================
fe_cfg = cfg.feature_engineering

print("=" * 70)
print(f"FEATURE ENGINEERING (Profile: {cfg._profile})")
print("=" * 70)
print(f"  Audit: {fe_cfg.audit}")
print(f"  Plot: {fe_cfg.plot}")
print(f"  Reference Date: {fe_cfg.reference_date}")
print("=" * 70)

df, rep = build_features(
    df, 
    reference_date=fe_cfg.reference_date,
    cvss_missing_fill=fe_cfg.cvss_missing_fill,
    epss_missing_fill=fe_cfg.epss_missing_fill,
    audit=fe_cfg.audit,
    plot=fe_cfg.plot,
    plot_top_missing=fe_cfg.plot_top_missing
)

# Display reports if audit is enabled
if rep is not None:
    print("\n📊 Missingness Report BEFORE feature engineering:")
    display(rep.before.head(20))
    
    print("\n📊 Missingness Report AFTER feature engineering:")
    display(rep.after.head(20))
    
    print("\n📊 Missingness DELTA (reductions):")
    display(rep.delta.head(20))
else:
    print(f"\n✅ Features built: {len(df):,} rows, {len(df.columns)} columns")

In [ ]:
# =============================================================================
# Normalize Features for Model Training (Optional)
# =============================================================================
print("=" * 70)
print("NORMALIZE FEATURES - MinMax Scaling for Model")
print("=" * 70)

feature_cols = get_default_feature_cols()
norm_df, params = normalize_features(df, feature_cols, method="minmax")

print(f"✅ Normalized {len(feature_cols)} features using MinMax scaling")
print(f"\nScaling parameters saved for reproducibility:")
for col, p in list(params.items())[:5]:
    print(f"  {col}: min={p['min']:.4f}, max={p['max']:.4f}")
print(f"  ... and {len(params) - 5} more")

print("\n📊 Normalized Feature Statistics:")
display(norm_df[feature_cols].describe().T[['mean', 'std', 'min', 'max']].round(4))

## 5. Weak Label Construction with Confidence Scores

Core innovation: Labels are graded (0-3) with confidence weights reflecting trustworthiness.

In [ ]:
# Build weak labels using modular function
df = build_weak_labels(df)

print(f"Weak labels constructed for {len(df):,} CVEs")
print(f"Columns added: soft_label, label_confidence, label_source")

# Print diagnostics
print_label_diagnostics(df)

## 6. Temporal Split (Train/Val/Test)

Temporal split simulates real-world deployment: train on historical, predict future.

In [ ]:
# Force reload temporal module to pick up timezone fix
import importlib
import src.utils.temporal
importlib.reload(src.utils.temporal)
from src.utils.temporal import make_temporal_splits

# Create temporal splits using modular function
train_df, val_df, test_df = make_temporal_splits(
    df, 
    date_col='published',
    split_date=SPLIT_DATE, 
    val_weeks=VAL_WEEKS
)

# Show label distribution in each split
print("\nLabel distribution by split:")
for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    if len(split_df) > 0:
        dist = split_df['soft_label'].value_counts().sort_index()
        print(f"  {name:5s}: {dict(dist)}")

## 7. Model Training

### 7.1 Confidence-Weighted LambdaRank (Primary Model)

In [ ]:
# Train LambdaRank using modular function
model = train_lambdarank(train_df, val_df, FEATURE_COLS, random_seed=RANDOM_SEED)

# Save model
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model, f)
print(f"\nModel saved to: {MODEL_PATH}")

### 7.2 Comparison Models (Optional - For Research Paper)

For thesis evaluation, we also train:
- DiffusionRank: Probabilistic label imputation
- RGCN: Graph neural network on CVE-vendor relations
- Bootstrap Ensemble: Uncertainty-aware ranking

**Note**: These require the existing model implementations in `src/models/`

In [ ]:
# Comparison models - only run if needed for research
RUN_COMPARISON_MODELS = False  # Set to True for full comparison study

if RUN_COMPARISON_MODELS:
    print("\n" + "=" * 70)
    print("TRAINING COMPARISON MODELS")
    print("=" * 70)
    
    from src.models.diffusion_imputer import DiffusionRankImputer
    from src.models.rgcn_ranker import RGCNRanker
    from src.models.bootstrap_ensemble import BootstrapEnsemble
    
    # DiffusionRank
    print("\n[1/3] DiffusionRank...")
    diffusion_model = DiffusionRankImputer(
        input_dim=len(FEATURE_COLS),
        hidden_dim=cfg.rgcn.hidden_channels,  # Use config
        num_steps=10,
        device=cfg.training_device
    )
    X_train, y_train, w_train, _, _ = prepare_ranking_data(train_df, FEATURE_COLS)
    diffusion_model.train_quick(X_train, y_train, epochs=50, batch_size=512)
    print("  DiffusionRank trained ✓")
    
    # RGCN
    print("\n[2/3] RGCN...")
    rgcn_model = RGCNRanker(
        num_features=len(FEATURE_COLS),
        hidden_dim=cfg.rgcn.hidden_channels,  # Use config
        num_relations=3,
        device=cfg.training_device
    )
    rgcn_model.train(train_df, val_df, FEATURE_COLS, epochs=cfg.rgcn.epochs)
    print("  RGCN trained ✓")
    
    # Bootstrap Ensemble
    print("\n[3/3] Bootstrap Ensemble...")
    ensemble_model = BootstrapEnsemble(n_estimators=10, random_state=RANDOM_SEED)
    ensemble_model.train(train_df, FEATURE_COLS)
    print("  Ensemble trained ✓")
    
    print("\nComparison models ready for evaluation")
else:
    print("\nSkipping comparison models (set RUN_COMPARISON_MODELS=True to enable)")

## 8. Evaluation on Test Set

### 8.1 Generate Predictions

In [ ]:
# Prepare test data
X_test = test_df[FEATURE_COLS].values

# Primary model predictions
test_df['ltr_score'] = model.predict(X_test)

# Baseline predictions
test_df['cvss_score'] = compute_cvss_only_scores(test_df)
test_df['heuristic_score'] = compute_heuristic_scores(test_df)
test_df['legacy_score'] = compute_legacy_label_scores(test_df)

print("Predictions generated for all models:")
print(f"  - LTR (confidence-weighted): {test_df['ltr_score'].notna().sum():,}")
print(f"  - CVSS baseline: {test_df['cvss_score'].notna().sum():,}")
print(f"  - Heuristic baseline: {test_df['heuristic_score'].notna().sum():,}")
print(f"  - Legacy label baseline: {test_df['legacy_score'].notna().sum():,}")

### 8.2 Compute Ranking Metrics

In [ ]:
# Evaluate all models
print("\n" + "=" * 70)
print("COMPARATIVE EVALUATION ON TEST SET")
print("=" * 70)

models = {
    'LTR (Confidence-Weighted)': 'ltr_score',
    'CVSS-only': 'cvss_score',
    'Weighted Heuristic': 'heuristic_score',
    'Legacy Labels': 'legacy_score'
}

results = {}
for model_name, score_col in models.items():
    metrics = evaluate_ranking(test_df, score_col, label_col='soft_label', 
                              group_col='published_week', k_values=[5, 10, 20])
    results[model_name] = metrics
    print(f"\n{model_name}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")

# Create comparison DataFrame
comparison_df = pd.DataFrame(results).T
print("\n" + "=" * 70)
print("MODEL COMPARISON TABLE")
print("=" * 70)
print(comparison_df.round(4))

# Compute improvements
baseline_ndcg10 = results['CVSS-only']['NDCG@10']
ltr_ndcg10 = results['LTR (Confidence-Weighted)']['NDCG@10']
improvement = 100 * (ltr_ndcg10 - baseline_ndcg10) / baseline_ndcg10 if baseline_ndcg10 > 0 else 0
print(f"\nImprovement over CVSS-only: +{improvement:.1f}% NDCG@10")

## 9. Explainability Analysis

### 9.1 Feature Importance

In [ ]:
# Plot feature importance (both gain and split count)
plot_feature_importance_comparison(model, FEATURE_COLS, height=600)

### 9.2 SHAP Explainability

In [ ]:
# SHAP analysis on test set sample
shap_values = plot_shap_summary(
    model, 
    test_df[FEATURE_COLS], 
    feature_names=FEATURE_COLS,
    max_display=15,
    sample_size=5000,
    random_seed=RANDOM_SEED
)

### 9.3 Individual CVE Explanations

In [ ]:
# Show detailed explanations for diverse example CVEs
explain_individual_predictions(
    test_df,
    score_col='ltr_score',
    feature_cols=FEATURE_COLS,
    label_col='soft_label',
    confidence_col='label_confidence',
    label_source_col='label_source',
    model=model,
    n_examples=5,
    random_seed=RANDOM_SEED
)

## 10. Results Summary & Conclusions

In [ ]:
print("\n" + "=" * 70)
print("FINAL SUMMARY: CONFIDENCE-WEIGHTED WEAK SUPERVISION")
print("=" * 70)

print("""
RESEARCH INNOVATION:
  Traditional LTR treats all labels equally. Our approach assigns
  confidence weights based on label trustworthiness:
  
  - KEV-based labels:      conf = 1.0  (confirmed exploitation)
  - High EPSS labels:      conf = 0.75 (statistical evidence)
  - Medium EPSS labels:    conf = 0.55 (moderate evidence)
  - ATT&CK/Healthcare:     +0.10 bonus (domain relevance)
  - CVSS/Recency-only:     conf <= 0.40 (noisy proxies)

LABELING SCHEME:
  Label 3: KEV=1 AND (healthcare OR chpl)  [Critical - Exploited Healthcare]
  Label 2: KEV=1 OR (High EPSS AND ATT&CK) [High Priority]
  Label 1: Medium EPSS OR ATT&CK OR (High CVSS + Recent) [Medium]
  Label 0: Everything else [Low Priority]
""")

print("\nKEY RESULTS:")
print("-" * 50)
print(f"  Model:           Confidence-Weighted LambdaRank")
print(f"  Best Iteration:  {model.best_iteration}")
print(f"  Val NDCG@10:     {model.best_score['valid']['ndcg@10']:.4f}")
print(f"\n  Test Set Performance:")
for metric, value in results['LTR (Confidence-Weighted)'].items():
    print(f"    {metric}: {value:.4f}")

print("\nIMPROVEMENT SUMMARY:")
print("-" * 50)
cvss_ndcg10 = results['CVSS-only']['NDCG@10']
ltr_ndcg10 = results['LTR (Confidence-Weighted)']['NDCG@10']
if cvss_ndcg10 > 0:
    print(f"  vs CVSS-only:      +{100*(ltr_ndcg10 - cvss_ndcg10)/cvss_ndcg10:.1f}% NDCG@10")

heur_ndcg10 = results['Weighted Heuristic']['NDCG@10']
if heur_ndcg10 > 0:
    print(f"  vs Weighted Heur:  +{100*(ltr_ndcg10 - heur_ndcg10)/heur_ndcg10:.1f}% NDCG@10")

print(f"\nMODEL SAVED: {MODEL_PATH}")
print(f"Dataset: {len(df):,} CVEs ({train_df['published'].min().date()} to {test_df['published'].max().date()})")
print(f"GPU Device: {device_manager.device}")

print("\n" + "=" * 70)
print("END OF NOTEBOOK - Ready for Production Deployment")
print("=" * 70)